# Pelatihan Model AI e-Sign (Prophet)
Notebook ini dirancang untuk dijalankan di **Google Colab**. 
Silakan **Upload file CSV historis pegawai** ke dalam direktori `/content` di Google Colab sebelum menjalankan script ini.

*Pastikan Anda sudah mengklik tombol **Connect / TPP** di kanan atas Colab untuk mendapatkan alokasi memori/GPU.*

In [ ]:
!pip install prophet statsmodels scikit-learn matplotlib

## 1. Import Library & Fungsi Pelatihan

In [ ]:
import pandas as pd
from prophet import Prophet
import json
from prophet.serialize import model_to_json
import os

def train_and_save_models():
    # Ubah sesuai dengan ID pegawai yang ada di CSV Anda yang di-upload
    pegawai_ids = [
        '1a2394378ddbb186f6f622e8b3c872ee6872623f',
        '54a8b8362ebcb16af08c8acf33a2d8d5f335cf5e',
        'ae914d89870f2450ca4c6ca9f34e3080317546e5',
        '8018dcd81ff171aa9629c08d95422c20e45e307b',
        'a4adb04d8392abc79d52ea247fabd8348b97a78a',
    ]
    
    current_dir = '/content'
    
    for pegawai_id in pegawai_ids:
        data_path = os.path.join(current_dir, f'historical_data_pegawai_{pegawai_id}.csv')
        if not os.path.exists(data_path):
            print(f"Data untuk pegawai {pegawai_id} tidak ditemukan. Melewati...")
            continue
            
        df = pd.read_csv(data_path)
        
        # --- TRAINING MODEL NODIN ---
        print(f"Melatih model Nodin untuk Pegawai {pegawai_id}...")
        df_nodin = df[['tanggal', 'nodin']].rename(columns={'tanggal': 'ds', 'nodin': 'y'})
        model_nodin = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        model_nodin.fit(df_nodin)
        
        with open(os.path.join(current_dir, f'model_nodin_pegawai_{pegawai_id}.json'), 'w') as f:
            json.dump(model_to_json(model_nodin), f)
            
        # --- TRAINING MODEL SPT ---
        print(f"Melatih model SPT untuk Pegawai {pegawai_id}...")
        df_spt = df[['tanggal', 'spt']].rename(columns={'tanggal': 'ds', 'spt': 'y'})
        model_spt = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
        model_spt.fit(df_spt)
        
        with open(os.path.join(current_dir, f'model_spt_pegawai_{pegawai_id}.json'), 'w') as f:
            json.dump(model_to_json(model_spt), f)

    print("Model AI (Prophet) berhasil dilatih dan disimpan! Silakan download file .json yang dihasilkan dari folder /content di menu sebelah kiri Colab.")


In [ ]:
# Eksekusi Pelatihan
train_and_save_models()

## 2. Evaluasi Model (Perbandingan MAE & RMSE)
Script di bawah ini untuk menampilkan evaluasi dan generate grafik perbandingan untuk laporan Anda.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

def evaluate_models():
    pegawai_id = '1a2394378ddbb186f6f622e8b3c872ee6872623f'
    data_path = f'/content/historical_data_pegawai_{pegawai_id}.csv'
    
    if not os.path.exists(data_path):
        print("Upload CSV terlebih dahulu!")
        return
        
    df = pd.read_csv(data_path)
    df_eval = df[['tanggal', 'nodin']].rename(columns={'tanggal': 'ds', 'nodin': 'y'})
    df_eval['ds'] = pd.to_datetime(df_eval['ds'])
    df_eval = df_eval.sort_values('ds')
    
    train_size = int(len(df_eval) * 0.8)
    train_df = df_eval.iloc[:train_size].copy()
    test_df = df_eval.iloc[train_size:].copy()
    
    predictions = pd.DataFrame({'ds': test_df['ds'], 'Aktual': test_df['y']})

    print("Melatih model Prophet...")
    model_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
    model_prophet.fit(train_df)
    future = model_prophet.make_future_dataframe(periods=len(test_df))
    pred_prophet = model_prophet.predict(future)['yhat'].iloc[-len(test_df):].values
    predictions['Prophet'] = np.maximum(0, pred_prophet)
    
    print("Melatih model ARIMA...")
    try:
        model_arima_fit = ARIMA(train_df['y'].values, order=(5,1,0)).fit()
        predictions['ARIMA'] = np.maximum(0, model_arima_fit.forecast(steps=len(test_df)))
    except:
        predictions['ARIMA'] = 0

    print("Melatih model Exponential Smoothing...")
    try:
        model_hw_fit = ExponentialSmoothing(train_df['y'].values, trend='add').fit()
        predictions['Exponential Smoothing'] = np.maximum(0, model_hw_fit.forecast(steps=len(test_df)))
    except:
        predictions['Exponential Smoothing'] = 0

    predictions['Naive Baseline'] = [train_df['y'].iloc[-1]] * len(test_df)

    models_to_eval = ['Prophet', 'ARIMA', 'Exponential Smoothing', 'Naive Baseline']
    
    print("\n--- HASIL EVALUASI MODEL ---")
    print(f"{'Model':<25} | {'MAE':<10} | {'RMSE'}")
    print("-" * 50)
    for m in models_to_eval:
        mae = mean_absolute_error(predictions['Aktual'], predictions[m])
        rmse = np.sqrt(mean_squared_error(predictions['Aktual'], predictions[m]))
        print(f"{m:<25} | {mae:<10.4f} | {rmse:.4f}")

    plt.figure(figsize=(12, 6))
    plt.plot(train_df['ds'], train_df['y'], label='Data Latih (Aktual)', color='gray')
    plt.plot(predictions['ds'], predictions['Aktual'], label='Data Uji (Aktual)', color='black', linewidth=2)
    colors = ['blue', 'orange', 'green', 'red']
    for idx, m in enumerate(models_to_eval):
        plt.plot(predictions['ds'], predictions[m], label=f'Prediksi {m}', color=colors[idx], linestyle='--')
    plt.title(f'Perbandingan Prediksi Model untuk Nota Dinas', fontsize=14)
    plt.legend()
    plt.show()

evaluate_models()